## <span style="color:#b8860b">Goodreads Books Data ETL Pipeline</span>

A random-sampling ETL pipeline designed to collect Goodreads book data and stream structured datasets directly to the **Hugging Face Hub**.

### Architecture & Core Interfaces

The system follows a strict separation of concerns, where each core component exposes exactly **one public entry point**:

| Class | Public entry point | Responsibility |
|---|---|---|
| `BookExtractor` | `extract(target_count, tried_book_ids)` | Everything needed to go from "I need N more books" to "here are N structured records": builds and drives a queue-based, self-healing crawl, parses each page as it arrives, and assembles final records once a book's author/editions enrichment is complete. |
| `BookLoader` | `load(records, filename, known_book_ids)` | Batch, persist locally, push to the Hugging Face Hub, and track scraped IDs. |
| `BookRunner` | `run(sample_size)` | Orchestrates the other two: asks `BookLoader` how many books are already scraped, asks `BookExtractor` for however many more are needed, then hands the result back to `BookLoader` to persist in checkpointed batches. |

**Execution Note:** To trigger the complete end-to-end pipeline, invoke `BookRunner().run(sample_size=...)`

### Scale Design Principles (~500,000 Books)

Instead of relying on external loop wrappers calling repeated `crawler.run()` instances, the crawl architecture leverages **Crawlee's native concurrency and queue management**:

* **Unified Request Processing:** Book, author, and editions pages are labeled and routed into a single, persistent `RequestQueue`. The entire collection run takes place within one uninterrupted `crawler.run(seed_requests)` execution.

* **Self-Healing Mechanics:** Upon reaching any terminal outcome (successful extraction, no usable data, or fatal request failure) for a specific book ID, the handler immediately enqueues a new replacement seed ID into the live queue.

* **Rate Limiting vs. Local Scaling:**

  * `desired_concurrency` and `max_concurrency` govern **local** system hardware usage (CPU, memory, event loop latency) via Crawlee's `AutoscaledPool`. The pool scales dynamically without knowledge of remote host degradation or WAF policies.

  * `max_tasks_per_minute=60` (~1 request/sec) serves as the strict rate ceiling protecting the crawler against Goodreads' WAF rate limits and preventing IP blocks.

### Resilience, State Management & Recovery

* **Target Awareness:** `sample_size=N` denotes the global target dataset size. If existing progress already meets or exceeds `N`, the execution terminates immediately without redundant work.

* **Continuous Part Numbering:** Existing dataset metadata is retrieved from the Hugging Face Hub prior to execution, ensuring batch file part numbers increment continuously across separate runs.

* **Periodic Flushing:** Extracted records are committed to local Parquet files every `PARQUET_BATCH_SIZE` (default: 5,000 records) to mitigate data loss during unexpected crashes.

* **Deterministic URL Recovery:** Editions page links are built deterministically from `work_id` to prevent broken navigation links.

* **Tracking File:** Processed identifiers are tracked statefully in `book_ids.parquet`.

### <span style="color:#b8860b">Packages</span>

In [1]:
import asyncio
import json
import random
import os
import re
import sys
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import polars as pl
from bs4 import BeautifulSoup
from crawlee import ConcurrencySettings, Request
from crawlee.crawlers import BasicCrawlingContext, BeautifulSoupCrawler, BeautifulSoupCrawlingContext
from crawlee.storages import RequestQueue
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import EntryNotFoundError, RepositoryNotFoundError
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO", format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}")

load_dotenv()

True

### <span style="color:#b8860b">Configuration</span>

In [2]:
# --- Goodreads URL templates ------------------------------------------------
BASE_BOOK_URL = "https://www.goodreads.com/book/show/"
EDITIONS_URL_TEMPLATE = "https://www.goodreads.com/work/editions/{work_id}"

# --- Random book ID sampling -------------------------------------------------
MIN_BOOK_ID = 1                 # Known
MAX_BOOK_ID = 10_000_000        # Unknown, just an arbitrary max value
DEFAULT_SAMPLE_SIZE = 100       # A test with a 100-book crawl
MAX_RANDOM_ID_ATTEMPTS = 500    # Give up drawing a fresh ID after this many collisions

# --- Crawlee route labels -----------------------------------------------------
LABEL_BOOK = "book"
LABEL_AUTHOR = "author"
LABEL_EDITIONS = "editions"

# --- Crawler tuning -----------------------------------------------------------
MAX_REQUEST_RETRIES = 10
REQUEST_HANDLER_TIMEOUT_SECONDS = 60

# --- Concurrency & real request pacing -----------------------------------------
# desired_concurrency / max_concurrency govern LOCAL resource usage --
# Crawlee's AutoscaledPool scales toward max_concurrency based on CPU/memory/
# event-loop health, with NO awareness of how the remote site is responding.
# max_tasks_per_minute is the one setting that caps the REAL outgoing request
# rate against Goodreads, independent of local concurrency -- leaving it at
# its default of inf would let the pool scale toward max_concurrency almost
# immediately for a lightweight HTTP crawler, reproducing WAF throttling.
CONCURRENCY_SETTINGS = ConcurrencySettings(
    desired_concurrency=5,
    max_concurrency=10,
    max_tasks_per_minute=60,  # ~1 request/second, matching Goodreads' documented API tolerance
)

# --- Contributor stats text patterns -------------------------------------------
FOLLOWERS_PATTERN = re.compile(r"followers\s*\(([\d,]+)\)", re.I)
DISTINCT_WORKS_PATTERN = re.compile(r"([\d,]+)\s+distinct\s+works?", re.I)

# --- Catalog guideline violation -----------------------------------------------
CATALOG_GUIDELINE_PATTERN = re.compile(r"does\s+not\s+meet\s+our\s+catalog\s+guidelines", re.I)

# --- Editions count -------------------------------------------------------------
EDITIONS_HEADING_PATTERN = re.compile(r"\b([\d,]+)\s+editions?\b", re.I)
EDITIONS_TOTAL_PATTERN = re.compile(
    r"showing\s+(?:[\d,]+\s*-\s*[\d,]+\s+of\s+([\d,]+)|all\s+([\d,]+))",
    re.I,
)

# --- Apollo cache recursion -------------------------------------------------------
APOLLO_CACHE_MAX_DEPTH = 15  # generous ceiling against reference cycles; real payloads need ~8-10

# --- Local staging output -----------------------------------------------------------
OUTPUT_DIR = Path("data")
PARQUET_FILENAME_TEMPLATE = "books-part{part}.parquet"
PARQUET_FILENAME_PATTERN = re.compile(r"books-part(\d+)\.parquet$")
PARQUET_BATCH_SIZE = 5_000

# --- Hugging Face Hub sink -----------------------------------------------------------
HF_TOKEN = os.getenv("HF_TOKEN")
HF_REPO_ID = "pfaha/goodreads-books"
HF_REPO_TYPE = "dataset"
HF_RAW_DIR = "raw"

# --- Scraped-IDs tracker -----------------------------------------------------------
SCRAPED_IDS_FILENAME = "book_ids.parquet"

### <span style="color:#b8860b">BookExtractor</span>

Everything needed to turn "collect N more books" into structured records, in one class: request building, page parsing, per-run crawl state, and the queue-based self-healing crawl itself. 

Parsing and state tracking are private (`_parse_*`, `_Progress`) since nothing outside this class ever needs them.

Public entry point: `extract(target_count, tried_book_ids)`.

In [3]:
class BookExtractor:
    """Drives one queue-based, self-healing sampling crawl per call, from raw HTML to final records."""

    @dataclass(slots=True)
    class _Progress:
        """Mutable state shared across all route handlers for one crawl run.

        Attributes:
            target_count: Number of NEW valid book records to collect this run.
            tried_book_ids: Set tracking every ID drawn so far (this run + all prior runs).
            records: List holding fully assembled book record dicts.
            pending: Maps book_id -> in-progress record, awaiting author/editions enrichment.
            pending_editions_lookup: Maps editions_url -> book_id, so the editions handler
                knows which pending record to update.
            done_event: Async event set once target_count records are collected.
        """

        target_count: int
        tried_book_ids: set[str]
        records: list[dict[str, Any]] = field(default_factory=list)
        pending: dict[str, dict[str, Any]] = field(default_factory=dict)
        pending_editions_lookup: dict[str, str] = field(default_factory=dict)
        done_event: asyncio.Event = field(default_factory=asyncio.Event)

        @property
        def successes(self) -> int:
            """Return count of successfully collected records this run.

            Returns:
                Number of records collected so far in this run.
            """
            return len(self.records)

        @property
        def is_complete(self) -> bool:
            """Check whether this run's target has been met.

            Returns:
                True if collected records meet or exceed target_count.
            """
            return self.successes >= self.target_count

        def next_random_book_id(self) -> str | None:
            """Draw, reserve, and return one previously untried book ID.

            Returns:
                A unique random book ID string, or None if no untried ID was
                found within MAX_RANDOM_ID_ATTEMPTS.
            """
            for _ in range(MAX_RANDOM_ID_ATTEMPTS):
                candidate = str(random.randint(MIN_BOOK_ID, MAX_BOOK_ID))
                if candidate not in self.tried_book_ids:
                    self.tried_book_ids.add(candidate)
                    return candidate
            return None

        def add_record(self, record: dict[str, Any]) -> bool:
            """Store a completed record if capacity remains; signal completion if target is met.

            Args:
                record: The completed book record dict to store.

            Returns:
                True if the record was added, False if the run is already complete.
            """
            if self.is_complete:
                return False
            self.records.append(record)
            if self.is_complete:
                self.done_event.set()
            return True

    def __init__(self) -> None:
        self.logger = logger.bind(component="BookExtractor")

    # --- Parsing (private -- only ever called from this class's own route handlers) ---

    def _ms_epoch_to_iso(self, ms: int | None) -> str | None:
        """Convert an Apollo-cache millisecond-epoch timestamp to an ISO date.

        Uses timedelta arithmetic from the Unix epoch rather than 
        datetime.fromtimestamp, since the latter relies on the platform C
        runtime's localtime/gmtime functions -- on Windows specifically,
        this raises OSError: [Errno 22] Invalid argument for any date
        before January 2, 1970, which silently breaks every book first
        published before that year (a large fraction of any random sample
        across Goodreads' full catalog).

        Args:
            ms: Milliseconds since Unix epoch, or None/0 if missing.

        Returns:
            ISO 8601 date string, or None if ms is falsy.
        """
        if not ms:
            return None
        epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
        return (epoch + timedelta(milliseconds=ms)).date().isoformat()

    def _resolve_refs(self, state: dict[str, Any], node: Any, depth: int = 0) -> Any:
        """Recursively replace Apollo cache {"__ref": "Type:id"} pointers with the object they point to.

        Args:
            state: The full flat Apollo cache dict (apolloState).
            node: The current node being resolved (dict, list, or scalar).
            depth: Current recursion depth; guards against reference cycles.

        Returns:
            node with every __ref pointer replaced, or None past APOLLO_CACHE_MAX_DEPTH.
        """
        if depth > APOLLO_CACHE_MAX_DEPTH:
            return None
        if isinstance(node, dict):
            if list(node.keys()) == ["__ref"]:
                return self._resolve_refs(state, state.get(node["__ref"]), depth + 1)
            return {key: self._resolve_refs(state, value, depth + 1) for key, value in node.items()}
        if isinstance(node, list):
            return [self._resolve_refs(state, item, depth + 1) for item in node]
        return node

    def _load_apollo_state(self, soup: BeautifulSoup) -> dict[str, Any] | None:
        """Parse and return the raw apolloState dict from a page's __NEXT_DATA__ script tag.

        Args:
            soup: Parsed HTML of any Goodreads page using Next.js hydration.

        Returns:
            The raw (un-de-referenced) apolloState dict, or None if absent/malformed.
        """
        tag = soup.find("script", id="__NEXT_DATA__")
        if not tag or not tag.string:
            return None
        try:
            payload = json.loads(tag.string)
            return payload["props"]["pageProps"]["apolloState"]
        except (json.JSONDecodeError, KeyError, TypeError):
            return None

    def _find_book_entry(self, apollo_state: dict[str, Any], book_id: str) -> dict | None:
        """Find and resolve the Apollo cache Book entry matching a given book_id.

        Args:
            apollo_state: The full flat Apollo cache dict.
            book_id: The Goodreads numeric book ID to match against.

        Returns:
            The resolved (de-referenced) Book object dict, or None if not found.
        """
        raw_key = next(
            (
                key
                for key, value in apollo_state.items()
                if key.startswith("Book:")
                and isinstance(value, dict)
                and str(value.get("legacyId")) == str(book_id)
            ),
            None,
        )
        if raw_key is None:
            return None
        return self._resolve_refs(apollo_state, apollo_state[raw_key])

    def _extract_all_contributors(self, book: dict[str, Any]) -> list[dict[str, Any]]:
        """Extract every contributor (primary author + secondary contributors).

        Args:
            book: A resolved (de-referenced) Book object from the Apollo cache.

        Returns:
            List of dicts with name, url, and role for every contributor found.
        """
        contributors = []
        primary_edge = book.get("primaryContributorEdge") or {}
        primary_node = primary_edge.get("node") or {}
        if primary_node.get("name"):
            contributors.append({
                "name": primary_node.get("name"),
                "url": primary_node.get("webUrl"),
                "role": primary_edge.get("role") or "Author",
            })
        for edge in book.get("secondaryContributorEdges") or []:
            if not isinstance(edge, dict):
                continue
            node = edge.get("node") or {}
            name = node.get("name")
            if not name:
                continue
            contributors.append({"name": name, "url": node.get("webUrl"), "role": edge.get("role") or "Contributor"})
        return contributors

    def _parse_book_page(self, soup: BeautifulSoup, book_id: str) -> dict[str, Any]:
        """Parse a book page's __NEXT_DATA__ Apollo cache into a structured dict.

        editions_url is derived directly from the work's legacyId
        (deterministic Goodreads URL scheme), not the Apollo cache's
        work.editions.__ref -> webUrl chain, which is frequently
        unpopulated for books with few editions.

        Args:
            soup: Parsed HTML of a Goodreads book page.
            book_id: The Goodreads numeric book ID this page is for.

        Returns:
            Dict with every book field, editions_url, first_author_url,
            is_nonexistent, and catalog_guideline_violation.
        """
        page_text = soup.get_text(" ", strip=True)
        catalog_guideline_violation = bool(CATALOG_GUIDELINE_PATTERN.search(page_text))

        apollo_state = self._load_apollo_state(soup)
        book = self._find_book_entry(apollo_state, book_id) if apollo_state else None

        if book is None or not book.get("title"):
            return {
                "title": None, "contributors": [], "first_author": None, "first_author_url": None,
                "description": None, "genres": [], "series": [], "format": None, "pages": None,
                "language": None, "asin": None, "isbn10": None, "isbn13": None, "publisher": None,
                "publication_date": None, "average_rating": None, "ratings_count": None,
                "reviews_count": None, "editions_url": None, "work_id": None,
                "is_nonexistent": not catalog_guideline_violation,
                "catalog_guideline_violation": catalog_guideline_violation,
            }

        work = book.get("work") or {}
        details = book.get("details") or {}
        stats = work.get("stats") or {}
        work_id = work.get("legacyId")

        genres = [
            genre_edge["genre"]["name"]
            for genre_edge in (book.get("bookGenres") or [])
            if isinstance(genre_edge, dict) and genre_edge.get("genre", {}).get("name")
        ]
        series = [
            {"name": entry.get("series", {}).get("title"), "position": entry.get("userPosition")}
            for entry in (book.get("bookSeries") or [])
            if entry.get("series", {}).get("title")
        ]
        contributors = self._extract_all_contributors(book)
        first_author_url = contributors[0]["url"] if contributors else None

        description_html = book.get("description")
        description_text = (
            BeautifulSoup(description_html, "html.parser").get_text("\n").strip()
            if description_html
            else None
        )

        return {
            "title": book.get("title"),
            "contributors": contributors,
            "first_author": contributors[0]["name"] if contributors else None,
            "first_author_url": first_author_url,
            "description": description_text,
            "genres": genres,
            "series": series,
            "format": details.get("format"),
            "pages": details.get("numPages"),
            "language": (details.get("language") or {}).get("name"),
            "asin": details.get("asin"),
            "isbn10": details.get("isbn"),
            "isbn13": details.get("isbn13"),
            "publisher": details.get("publisher"),
            "publication_date": self._ms_epoch_to_iso(details.get("publicationTime")),
            "average_rating": stats.get("averageRating"),
            "ratings_count": stats.get("ratingsCount"),
            "reviews_count": stats.get("textReviewsCount"),
            "editions_url": EDITIONS_URL_TEMPLATE.format(work_id=work_id) if work_id else None,
            "work_id": work_id,
            "is_nonexistent": False,
            "catalog_guideline_violation": catalog_guideline_violation,
        }

    def _parse_author_page(self, soup: BeautifulSoup) -> dict[str, Any]:
        """Extract follower count and distinct-works count from an author page.

        Args:
            soup: Parsed HTML of a Goodreads author profile page.

        Returns:
            Dict with num_followers and num_distinct_works.
        """
        page_text = soup.get_text(" ", strip=True)
        followers_match = FOLLOWERS_PATTERN.search(page_text)
        works_match = DISTINCT_WORKS_PATTERN.search(page_text)
        return {
            "num_followers": int(followers_match.group(1).replace(",", "")) if followers_match else None,
            "num_distinct_works": int(works_match.group(1).replace(",", "")) if works_match else None,
        }

    def _parse_editions_page(self, soup: BeautifulSoup) -> int | None:
        """Extract the exact total edition count from an editions-listing page.

        Args:
            soup: Parsed HTML of a work's editions-listing page (first page).

        Returns:
            The exact total number of editions, or None if not found.
        """
        page_text = soup.get_text(" ", strip=True)
        match = EDITIONS_TOTAL_PATTERN.search(page_text)
        if match:
            return int((match.group(1) or match.group(2)).replace(",", ""))
        match = EDITIONS_HEADING_PATTERN.search(page_text)
        if match:
            return int(match.group(1).replace(",", ""))
        return None

    # --- Request building (private) ---

    def _build_book_request(self, book_id: str) -> Request:
        """Build a Crawlee request for a single book page.

        Args:
            book_id: The Goodreads book ID to target.

        Returns:
            Configured Request object for the book page, labeled LABEL_BOOK.
        """
        return Request.from_url(
            f"{BASE_BOOK_URL}{book_id}",
            label=LABEL_BOOK,
            user_data={"book_id": book_id},
            unique_key=f"book-{book_id}",
        )

    def _build_author_request(self, author_url: str, book_id: str) -> Request:
        """Build a Crawlee request for a book's first author page.

        Args:
            author_url: The full profile URL of the author.
            book_id: Associated book_id awaiting author enrichment.

        Returns:
            Configured Request object for the author page, labeled LABEL_AUTHOR.
        """
        return Request.from_url(
            author_url,
            label=LABEL_AUTHOR,
            user_data={"book_id": book_id},
            unique_key=f"author-{book_id}",
        )

    def _build_editions_request(self, editions_url: str, book_id: str) -> Request:
        """Build a Crawlee request for a work's editions-listing page.

        Args:
            editions_url: The full editions-listing URL for the book's work.
            book_id: Associated book_id awaiting editions-count enrichment.

        Returns:
            Configured Request object for the editions page, labeled LABEL_EDITIONS.
        """
        return Request.from_url(
            editions_url,
            label=LABEL_EDITIONS,
            user_data={"book_id": book_id},
            unique_key=f"editions-{book_id}",
        )

    def _require_book_id(self, request: Request) -> str:
        """Extract and validate book_id from a request's user_data.

        Args:
            request: The Crawlee request carrying book_id in its user_data.

        Returns:
            The validated string book_id.

        Raises:
            ValueError: If book_id is missing or not a string.
        """
        book_id = request.user_data.get("book_id")
        if not isinstance(book_id, str):
            raise ValueError(f"Missing or invalid book_id in request user_data: {book_id!r}")
        return book_id

    def _optional_book_id(self, request: Request) -> str | None:
        """Extract book_id from a request's user_data, if valid.

        Args:
            request: The Crawlee request carrying book_id in its user_data.

        Returns:
            The string book_id if present and valid, otherwise None.
        """
        book_id = request.user_data.get("book_id")
        return book_id if isinstance(book_id, str) else None

    # --- Record assembly (private) ---

    def _start_record(self, book_id: str, book_data: dict[str, Any]) -> dict[str, Any]:
        """Build the initial pending record for a book from its parsed book page.

        Tracks which enrichment steps (author, editions) are still
        outstanding via internal "_awaiting_*" flags, consumed by
        _is_ready_to_assemble and stripped by _assemble_record.

        Args:
            book_id: The Goodreads numeric book ID this record is for.
            book_data: The dict returned by _parse_book_page for this book.

        Returns:
            A pending record dict, not yet finalized.
        """
        record = {key: value for key, value in book_data.items() if key not in ("is_nonexistent", "first_author_url", "editions_url")}
        record["book_id"] = book_id
        record["url"] = f"{BASE_BOOK_URL}{book_id}"
        record["num_editions"] = None
        record["has_more_editions"] = False
        record["_awaiting_author"] = bool(book_data.get("first_author_url"))
        record["_awaiting_editions"] = bool(book_data.get("editions_url"))
        return record

    def _is_ready_to_assemble(self, record: dict[str, Any]) -> bool:
        """Check whether a pending record has no more enrichment steps outstanding.

        Args:
            record: The pending record dict to check.

        Returns:
            True if neither author nor editions enrichment is still pending.
        """
        return not record.get("_awaiting_author") and not record.get("_awaiting_editions")

    def _assemble_record(self, record: dict[str, Any]) -> dict[str, Any]:
        """Strip internal bookkeeping flags and return the final record, ready to persist.

        Args:
            record: A pending record dict that has passed _is_ready_to_assemble.

        Returns:
            The final record dict, with derived fields set and no
            leading-underscore internal keys remaining.
        """
        record.pop("_awaiting_author", None)
        record.pop("_awaiting_editions", None)
        record["is_part_of_series"] = bool(record.get("series"))
        return record

    # --- Crawl construction and routing (private) ---

    def _build_crawler(self) -> BeautifulSoupCrawler:
        """Construct a bare BeautifulSoupCrawler with global concurrency settings applied.

        Returns:
            An un-routed BeautifulSoupCrawler instance.
        """
        return BeautifulSoupCrawler(
            max_request_retries=MAX_REQUEST_RETRIES,
            request_handler_timeout=timedelta(seconds=REQUEST_HANDLER_TIMEOUT_SECONDS),
            retry_on_blocked=True,
            concurrency_settings=CONCURRENCY_SETTINGS,
            configure_logging=False,
        )

    def _attach_handlers(self, crawler: BeautifulSoupCrawler, progress: "BookExtractor._Progress") -> None:
        """Wire book/author/editions/failure handlers onto crawler, closing over progress.

        Args:
            crawler: The BeautifulSoupCrawler instance to attach routes to.
            progress: Shared mutable _Progress tracking this run's state.
        """

        async def _enqueue_replacement(context: BeautifulSoupCrawlingContext) -> None:
            """Enqueue one fresh random book ID unless the run is already complete."""
            if progress.is_complete:
                return None
            new_id = progress.next_random_book_id()
            if new_id is None:
                self.logger.warning("No untried book ID found within attempt limit -- ID space may be exhausted")
                return None
            await context.add_requests([self._build_book_request(new_id)])

        async def _try_finalize(context: BeautifulSoupCrawlingContext, book_id: str) -> None:
            """Assemble and store a pending record once it has no more enrichment steps outstanding."""
            record = progress.pending.get(book_id)
            if record is None or not self._is_ready_to_assemble(record):
                return None
            progress.pending.pop(book_id)
            final_record = self._assemble_record(record)
            if not progress.add_record(final_record):
                return None
            self.logger.info(f"book_id={book_id}: collected ({progress.successes}/{progress.target_count} target)")
            if not progress.is_complete:
                await _enqueue_replacement(context)

        @crawler.router.handler(LABEL_BOOK)
        async def handle_book(context: BeautifulSoupCrawlingContext) -> None:
            """Route handler for Goodreads book pages."""
            if progress.done_event.is_set():
                return None

            book_id = self._require_book_id(context.request)
            book_data = self._parse_book_page(context.soup, book_id)

            if book_data["is_nonexistent"]:
                await _enqueue_replacement(context)
                return None

            if progress.done_event.is_set():
                return None

            record = self._start_record(book_id, book_data)
            progress.pending[book_id] = record

            editions_url = book_data.get("editions_url")
            first_author_url = book_data.get("first_author_url")
            if editions_url:
                progress.pending_editions_lookup[editions_url] = book_id
                await context.add_requests([self._build_editions_request(editions_url, book_id)])
            if first_author_url:
                await context.add_requests([self._build_author_request(first_author_url, book_id)])

            await _try_finalize(context, book_id)

        @crawler.router.handler(LABEL_AUTHOR)
        async def handle_author(context: BeautifulSoupCrawlingContext) -> None:
            """Route handler for Goodreads author profile pages."""
            book_id = self._require_book_id(context.request)
            record = progress.pending.get(book_id)
            if record is None:
                return None

            author_stats = self._parse_author_page(context.soup)
            if record.get("contributors"):
                record["contributors"][0] = {**record["contributors"][0], **author_stats}
            record["_awaiting_author"] = False

            await _try_finalize(context, book_id)

        @crawler.router.handler(LABEL_EDITIONS)
        async def handle_editions(context: BeautifulSoupCrawlingContext) -> None:
            """Route handler for Goodreads work editions-listing pages."""
            book_id = self._require_book_id(context.request)
            progress.pending_editions_lookup.pop(context.request.url, None)

            record = progress.pending.get(book_id)
            if record is None:
                return None

            num_editions = self._parse_editions_page(context.soup)
            record["num_editions"] = num_editions
            record["has_more_editions"] = bool(num_editions and num_editions > 1)
            record["_awaiting_editions"] = False

            await _try_finalize(context, book_id)

        @crawler.failed_request_handler
        async def handle_failed(context: BeautifulSoupCrawlingContext | BasicCrawlingContext, error: Exception) -> None:
            """Handler for permanently failed Crawlee requests (e.g. exhausted retries, blocked)."""
            label = context.request.label
            book_id = self._optional_book_id(context.request)
            self.logger.warning(f"Request failed permanently: label={label} book_id={book_id} error={error}")

            bs_context = context  # type: ignore[assignment]

            if label == LABEL_BOOK:
                if book_id is not None:
                    progress.pending.pop(book_id, None)
                await _enqueue_replacement(bs_context) # type: ignore
                return None

            if book_id is None:
                return None

            record = progress.pending.get(book_id)
            if record is None:
                return None

            if label == LABEL_AUTHOR:
                record["_awaiting_author"] = False
            elif label == LABEL_EDITIONS:
                progress.pending_editions_lookup.pop(context.request.url, None)
                record["_awaiting_editions"] = False

            await _try_finalize(bs_context, book_id) # type: ignore

    async def _drive_to_completion(self, crawler: BeautifulSoupCrawler, seed_requests: list[Request], progress: "BookExtractor._Progress") -> None:
        """Run the crawler until naturally finished or the target is met, whichever first.

        Args:
            crawler: Configured BeautifulSoupCrawler instance.
            seed_requests: Initial batch of requests to seed the queue with.
            progress: Shared mutable _Progress containing target/done_event.
        """
        run_task = asyncio.create_task(crawler.run(seed_requests))
        done_wait_task = asyncio.create_task(progress.done_event.wait())

        done, pending = await asyncio.wait({run_task, done_wait_task}, return_when=asyncio.FIRST_COMPLETED)

        if done_wait_task in done and not run_task.done():
            run_task.cancel()
        for task in pending:
            task.cancel()

        try:
            await run_task
        except asyncio.CancelledError:
            pass

    async def _reset_request_queue_storage(self) -> None:
        """Drop any request-queue state left over from a previous asyncio event loop.

        Crawlee's file-system request queue client caches an asyncio.Lock
        bound to whatever event loop was active when first created. Since
        each notebook cell execution can run its own event loop, that lock
        can end up bound to a dead loop on repeated runs; dropping the
        queue forces a fresh client/lock on the next open().
        """
        queue = await RequestQueue.open()
        await queue.drop()

    async def extract(self, target_count: int, tried_book_ids: set[str]) -> list[dict[str, Any]]:
        """Run a queue-based sampling crawl until target_count new valid records are collected.

        This is the single public entry point BookRunner calls. Seeds
        desired_concurrency * 4 initial book requests (capped at
        target_count) so multiple books are in flight from the start,
        then lets Crawlee's own AutoscaledPool (bounded by
        CONCURRENCY_SETTINGS.max_tasks_per_minute) and the self-healing
        replacement handlers keep the queue fed until either the target
        is met or the queue naturally empties (ID space exhausted).

        Args:
            target_count: Number of NEW valid book records to collect.
            tried_book_ids: Set of book_ids to exclude (already scraped in
                prior runs); mutated in place with every ID drawn this run.

        Returns:
            List of up to target_count valid book record dicts.
        """
        if target_count <= 0:
            return []

        await self._reset_request_queue_storage()

        progress = self._Progress(target_count=target_count, tried_book_ids=tried_book_ids)

        seed_count = min(max(CONCURRENCY_SETTINGS.desired_concurrency * 4, 1), target_count)
        seed_ids = [progress.next_random_book_id() for _ in range(seed_count)]
        seed_requests = [self._build_book_request(book_id) for book_id in seed_ids if book_id is not None]
        if not seed_requests:
            self.logger.warning("Unable to seed crawl -- no untried book IDs available")
            return []

        crawler = self._build_crawler()
        self._attach_handlers(crawler, progress)

        await self._drive_to_completion(crawler, seed_requests, progress)

        if progress.successes < progress.target_count:
            self.logger.warning(
                f"Crawl ended early: {progress.successes}/{progress.target_count} collected "
                "(queue exhausted or ID space too sparse)"
            )

        return progress.records[:target_count]

## <span style="color:#b8860b">BookLoader</span>

Independent of everything above: knows nothing about Goodreads, HTML, or crawling -- only how to persist a list of record dicts and track which book_ids have already been seen. 

Batches records, persists locally, pushes to the Hugging Face Hub, and maintains `book_ids.parquet` for deduplication.

Public entry point: `load(records, filename, known_book_ids)`.

In [4]:
class BookLoader:
    """Persists book batches locally, pushes them to the Hub, and tracks scraped IDs."""

    def __init__(self) -> None:
        self.logger = logger.bind(component="BookLoader")
        self.api = self._get_hf_api()

    def _get_hf_api(self) -> HfApi:
        """Build an authenticated Hugging Face Hub API client.

        Returns:
            An HfApi instance authenticated with HF_TOKEN.

        Raises:
            RuntimeError: If HF_TOKEN is not set.
        """
        if not HF_TOKEN:
            raise RuntimeError("HF_TOKEN is not set -- check your .env file")
        return HfApi(token=HF_TOKEN)

    def _save_locally(self, df: pl.DataFrame, filename: str) -> Path:
        """Write a DataFrame to a local Parquet file under OUTPUT_DIR.

        Args:
            df: The DataFrame to persist.
            filename: File name only (e.g. "books-part7.parquet").

        Returns:
            The full local path the file was written to.
        """
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        local_path = OUTPUT_DIR / filename
        df.write_parquet(local_path)
        return local_path

    def _upload(self, local_path: Path, filename: str, commit_message: str) -> None:
        """Upload a local file to {HF_REPO_ID}/{HF_RAW_DIR}/{filename}.

        Args:
            local_path: Local path of the file to upload.
            filename: File name only, used to build the path in the repo.
            commit_message: Commit message for this upload.
        """
        self.api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=f"{HF_RAW_DIR}/{filename}",
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=commit_message,
        )

    def get_next_part_number(self) -> int:
        """Determine the next books-part number to use, continuing from the Hub.

        Returns:
            1 if no books-part*.parquet files exist yet, otherwise the
            highest existing part number + 1.
        """
        existing_files = self.api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
        part_numbers = [
            int(match.group(1))
            for f in existing_files
            if (match := PARQUET_FILENAME_PATTERN.search(f)) and f.startswith(f"{HF_RAW_DIR}/")
        ]
        next_part = max(part_numbers, default=0) + 1
        self.logger.info(f"Next part number: {next_part} ({len(part_numbers)} existing part(s) found)")
        return next_part

    def load_scraped_ids(self) -> set[str]:
        """Load the set of book_ids already scraped in any previous run.

        Returns:
            Set of book_id strings already present in the tracker file, or
            an empty set if the tracker does not exist yet.
        """
        try:
            local_path = hf_hub_download(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                filename=f"{HF_RAW_DIR}/{SCRAPED_IDS_FILENAME}",
                token=HF_TOKEN,
            )
        except (EntryNotFoundError, RepositoryNotFoundError):
            self.logger.info("No book_ids tracker found on the Hub yet, starting fresh")
            return set()

        scraped_ids = set(pl.read_parquet(local_path).get_column("book_id").to_list())
        self.logger.info(f"Loaded {len(scraped_ids)} already-scraped book_id(s) from the tracker")
        return scraped_ids

    def _update_scraped_ids(self, new_book_ids: list[str], known_book_ids: set[str]) -> set[str]:
        """Merge new book_ids into the tracker, save locally, and push to the Hub.

        Args:
            new_book_ids: book_ids collected in the batch just loaded.
            known_book_ids: Full set of book_ids already known before this batch.

        Returns:
            The updated, merged set of all known book_ids.
        """
        updated_ids = known_book_ids | set(new_book_ids)
        df = pl.DataFrame({"book_id": sorted(updated_ids)})
        local_path = self._save_locally(df, SCRAPED_IDS_FILENAME)
        self._upload(local_path, SCRAPED_IDS_FILENAME, f"Update book_ids tracker ({len(updated_ids)} total)")
        return updated_ids

    def load(self, records: list[dict[str, Any]], filename: str, known_book_ids: set[str]) -> tuple[pl.DataFrame, set[str]]:
        """Batch records into a DataFrame, save and push it, then update the scraped-ids tracker.

        Args:
            records: List of final book record dicts to persist.
            filename: File name only for the data batch (e.g. "books-part7.parquet").
            known_book_ids: Full set of book_ids already known before this batch.

        Returns:
            Tuple of (the DataFrame written and uploaded, the updated set
            of all known book_ids including this batch).
        """
        df = pl.DataFrame(records)
        local_path = self._save_locally(df, filename)

        self.logger.info(f"Sinking {HF_RAW_DIR}/{filename}: uploading {df.height} row(s)...")
        self._upload(local_path, filename, f"Add {filename} ({df.height} books)")
        self.logger.info(f"Sunk {HF_RAW_DIR}/{filename}: {df.height} row(s) pushed to {HF_REPO_ID}")

        updated_ids = self._update_scraped_ids(df.get_column("book_id").to_list(), known_book_ids)
        self.logger.info(f"Tracker updated: {len(updated_ids)} book_id(s) now recorded")

        return df, updated_ids

### <span style="color:#b8860b">BookRunner</span>

Ties `BookExtractor` and `BookLoader` together: asks `BookLoader` how many books are already scraped, asks `BookExtractor` for however many more are needed, then hands the result back to `BookLoader` to persist in checkpointed batches.

Public entry point: `run(sample_size)`.

In [5]:
class BookRunner:
    """Orchestrates BookExtractor and BookLoader to hit a sample_size target."""

    def __init__(self) -> None:
        self.logger = logger.bind(component="BookRunner")
        self.extractor = BookExtractor()
        self.loader = BookLoader()

    async def run(self, sample_size: int = DEFAULT_SAMPLE_SIZE) -> pl.DataFrame:
        """Run the ETL pipeline until the dataset holds sample_size valid book records total.

        sample_size is a TARGET, not an increment: if the tracker already
        holds sample_size or more book_ids, this returns immediately.
        Otherwise it runs one queue-based sampling crawl for the
        remaining count, then flushes results to the Hub in
        PARQUET_BATCH_SIZE chunks.

        Args:
            sample_size: Target total number of valid book records the
                dataset should hold.

        Returns:
            A DataFrame of the records collected in THIS run (empty if the
            target was already met before this call).
        """
        self.logger.info(f"Starting pipeline: target {sample_size} total valid book(s)...")

        known_book_ids = self.loader.load_scraped_ids()

        if len(known_book_ids) >= sample_size:
            self.logger.info(f"Target already met: {len(known_book_ids)}/{sample_size} book(s) already scraped, nothing to do")
            return pl.DataFrame()

        remaining_target = sample_size - len(known_book_ids)
        self.logger.info(f"{len(known_book_ids)} book(s) already scraped, collecting {remaining_target} more")

        part_number = self.loader.get_next_part_number()
        tried_book_ids: set[str] = set(known_book_ids)

        new_records = await self.extractor.extract(remaining_target, tried_book_ids)

        all_batches: list[pl.DataFrame] = []
        for start in range(0, len(new_records), PARQUET_BATCH_SIZE):
            batch = new_records[start : start + PARQUET_BATCH_SIZE]
            batch_df, known_book_ids = self.loader.load(batch, PARQUET_FILENAME_TEMPLATE.format(part=part_number), known_book_ids)
            all_batches.append(batch_df)
            part_number += 1

        self.logger.info(f"Pipeline complete: {len(known_book_ids)}/{sample_size} total book(s) now in the dataset")

        return pl.concat(all_batches, how="diagonal_relaxed") if all_batches else pl.DataFrame()

### <span style="color:#b8860b">Run</span>

Only this cell needs to be called to run the whole pipeline. Safe to re-run repeatedly with the same `sample_size`. 

For the real 500,000-book run, call with `sample_size=500_000` and let it run continuously; `PARQUET_BATCH_SIZE` checkpoints every 5,000 books collected, and `book_ids.parquet` is updated after every checkpoint, so an interrupted run resumes cleanly.

In [6]:
result_df = await BookRunner().run(sample_size=DEFAULT_SAMPLE_SIZE)

22:51:18 | INFO     | Starting pipeline: target 100 total valid book(s)...
22:51:18 | INFO     | Loaded 100 already-scraped book_id(s) from the tracker
22:51:18 | INFO     | Target already met: 100/100 book(s) already scraped, nothing to do


In [7]:
with pl.Config(
    tbl_cols=-1,                # show all columns, no truncation
    tbl_rows=-1,                # show all rows requested (not needed for head(2), but harmless)
    fmt_str_lengths=1_000,      # show up to 1,000 chars per string cell before truncating
    tbl_width_chars=-1,         # don't wrap/truncate based on terminal width
):
    print(result_df.head())

shape: (0, 0)
┌┐
╞╡
└┘
